# 🌍 CDM 2026 — Exploratory Data Analysis (EDA)

> **Objectif** : Comprendre nos données avant de modéliser.  
> On explore les 4 fichiers CSV, on détecte les problèmes, on extrait les insights clés.

---
**Auteur** : Dylan Bayom  
**Dataset** : International Football Results 1872–2026  
**Fichiers** : results.csv · goalscorers.csv · shootouts.csv · former_names.csv

## 📦 0. Imports et configuration

On importe toutes les bibliothèques nécessaires pour l'analyse.

In [ ]:
# ── Manipulation des données ──────────────────────────
import pandas as pd       # lire et manipuler les CSV
import numpy as np        # calculs numériques

# ── Visualisation ─────────────────────────────────────
import matplotlib.pyplot as plt   # graphiques classiques
import seaborn as sns             # graphiques statistiques plus jolis
import plotly.express as px       # graphiques interactifs

# ── Configuration de l'affichage ──────────────────────
pd.set_option('display.max_columns', None)   # afficher toutes les colonnes
pd.set_option('display.max_rows', 50)        # afficher jusqu'à 50 lignes
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 décimales

# Style des graphiques matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Blues_r')

print('✅ Imports OK')

---
## 📂 1. Chargement des données

On lit les 4 fichiers CSV depuis le dossier `data/`.  
On parse les dates directement à la lecture pour éviter de le faire plus tard.

In [ ]:
# ── Lire les 4 fichiers CSV ───────────────────────────
# parse_dates=['date'] = convertir la colonne date en format datetime automatiquement

results     = pd.read_csv('data/results.csv',     parse_dates=['date'])
goals       = pd.read_csv('data/goalscorers.csv', parse_dates=['date'])
shootouts   = pd.read_csv('data/shootouts.csv',   parse_dates=['date'])
former      = pd.read_csv('data/former_names.csv')

print('✅ Données chargées')
print(f'   results     : {len(results):,} lignes')
print(f'   goals       : {len(goals):,} lignes')
print(f'   shootouts   : {len(shootouts):,} lignes')
print(f'   former_names: {len(former):,} lignes')

---
## 🔎 2. Structure des données

On regarde la forme de chaque fichier : colonnes, types, premières lignes.  
C'est la première chose à faire pour comprendre ce qu'on a.

In [ ]:
# ── Structure de results.csv ──────────────────────────
# C'est le fichier principal — une ligne = un match joué

print('=== RESULTS.CSV ===')
print(f'Forme : {results.shape}')   # (nb_lignes, nb_colonnes)
print(f'Période : {results.date.min().date()} → {results.date.max().date()}')
print()
print('Types des colonnes :')
print(results.dtypes)
print()
print('5 premières lignes :')
results.head()

In [ ]:
# ── Structure de goalscorers.csv ──────────────────────
# Une ligne = un but marqué dans un match
# Contient : buteur, minute, si c'est un penalty ou un CSC

print('=== GOALSCORERS.CSV ===')
print(f'Forme : {goals.shape}')
print()
print('Types des colonnes :')
print(goals.dtypes)
print()
print('5 premières lignes :')
goals.head()

In [ ]:
# ── Structure de shootouts.csv ────────────────────────
# Une ligne = une séance de tirs au but
# Complète results.csv pour les matchs K.O. qui finissent nuls

print('=== SHOOTOUTS.CSV ===')
print(f'Forme : {shootouts.shape}')
print()
print('Types des colonnes :')
print(shootouts.dtypes)
print()
print('5 premières lignes :')
shootouts.head()

In [ ]:
# ── Structure de former_names.csv ─────────────────────
# Dictionnaire des anciens noms de pays
# Ex : Zaïre → DR Congo, URSS → Russie
# Permet d'éviter de traiter la même équipe comme deux équipes différentes

print('=== FORMER_NAMES.CSV ===')
print(f'Forme : {former.shape}')
print()
former

---
## ❓ 3. Valeurs manquantes (NaN)

On cherche les valeurs manquantes dans chaque fichier.  
Un NaN = une case vide dans le tableau.  
Chaque NaN doit être compris : est-ce un problème ou un cas normal ?

In [ ]:
# ── NaN dans results.csv ──────────────────────────────
print('=== NaN dans results.csv ===')
nan_results = results.isnull().sum()
print(nan_results)
print()

# On regarde quels matchs ont des scores manquants
nan_scores = results[results.home_score.isna()]
print(f'Matchs sans score : {len(nan_scores)}')
print('Tournois concernés :')
print(nan_scores.tournament.value_counts())
print()
# → Ces 72 lignes sont les matchs CDM 2026 à prédire !

In [ ]:
# ── NaN dans goalscorers.csv ──────────────────────────
print('=== NaN dans goalscorers.csv ===')
print(goals.isnull().sum())
print()
# scorer NaN = on sait qu'il y a eu un but mais pas qui l'a marqué
# minute NaN = on ne sait pas à quelle minute le but a été marqué
# → Non bloquant : on n'a pas besoin du nom du buteur pour nos features
# → Pour les features de minutes, on filtrera sur minute.notna()

In [ ]:
# ── NaN dans shootouts.csv ────────────────────────────
print('=== NaN dans shootouts.csv ===')
print(shootouts.isnull().sum())
# first_shooter NaN = on ne sait pas quelle équipe a tiré en premier
# → Non bloquant : on utilise surtout la colonne 'winner'

---
## 📊 4. Distribution des résultats

On regarde comment les matchs se répartissent entre victoire domicile, nul et victoire extérieur.  
C'est la **variable cible** (target) de notre modèle ML.

In [ ]:
# ── Créer la colonne résultat ─────────────────────────
# On filtre post-1990 et on ignore les matchs sans score (CDM 2026)

r90 = results[
    (results.date.dt.year >= 1990) &    # garder seulement les matchs modernes
    (results.home_score.notna())         # ignorer les 72 matchs CDM 2026 sans score
].copy()

# Créer la colonne 'result' : qui a gagné ?
r90['result'] = np.where(
    r90.home_score > r90.away_score, 'home_win',   # l'équipe domicile a marqué plus
    np.where(
        r90.home_score < r90.away_score, 'away_win',  # l'équipe extérieur a marqué plus
        'draw'                                          # même nombre de buts = nul
    )
)

print(f'Matchs post-1990 : {len(r90):,}')
print()
print('Distribution des résultats :')
print(r90.result.value_counts())
print()
print('En pourcentage :')
print(r90.result.value_counts(normalize=True).round(3) * 100)

In [ ]:
# ── Graphique : distribution des résultats ────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Graphique 1 : camembert
counts = r90.result.value_counts()
labels = ['Victoire domicile', 'Victoire extérieur', 'Match nul']
colors = ['#1F4E79', '#E74C3C', '#95A5A6']
axes[0].pie(
    counts.values,
    labels=labels,
    colors=colors,
    autopct='%1.1f%%',
    startangle=90
)
axes[0].set_title('Distribution des résultats (post-1990)', fontsize=13, fontweight='bold')

# Graphique 2 : barres
axes[1].bar(labels, counts.values, color=colors, edgecolor='white')
axes[1].set_title('Nombre de matchs par résultat', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Nombre de matchs')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 100, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('data/plot_distribution_resultats.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight : La classe "draw" est minoritaire → attention au déséquilibre des classes')

---
## 🏟️ 5. Effet du terrain neutre

En CDM 2026, tous les matchs se jouent sur terrain neutre (USA/Canada/Mexique).  
On vérifie si ça change vraiment les résultats.

In [ ]:
# ── Terrain neutre vs terrain réel ────────────────────
# neutral=False = l'équipe 'home' joue vraiment à domicile
# neutral=True  = terrain neutre, 'home' est juste l'équipe listée en premier

print('=== EFFET DU TERRAIN NEUTRE ===')
print()

for neutral_val, label in [(False, 'Terrain réel (domicile)'), (True, 'Terrain neutre')]:
    subset = r90[r90.neutral == neutral_val]
    dist = subset.result.value_counts(normalize=True).round(3)
    print(f'{label} ({len(subset):,} matchs) :')
    print(f'  Victoire home : {dist.get("home_win", 0):.1%}')
    print(f'  Match nul     : {dist.get("draw", 0):.1%}')
    print(f'  Victoire away : {dist.get("away_win", 0):.1%}')
    print()

print('💡 Insight : Sur terrain neutre, l\'avantage domicile tombe de 50.7% à 42.6%')
print('   → La colonne neutral sera une feature importante pour notre modèle !')

In [ ]:
# ── Graphique : terrain neutre vs domicile réel ───────
fig, ax = plt.subplots(figsize=(10, 5))

categories = ['Victoire home', 'Match nul', 'Victoire away']
domicile   = [50.7, 23.5, 25.8]   # terrain réel
neutre     = [42.6, 23.7, 33.7]   # terrain neutre

x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width/2, domicile, width, label='Terrain réel',   color='#1F4E79', alpha=0.85)
bars2 = ax.bar(x + width/2, neutre,   width, label='Terrain neutre', color='#E74C3C', alpha=0.85)

ax.set_ylabel('Pourcentage (%)', fontsize=12)
ax.set_title('Impact du terrain neutre sur les résultats', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 60)

# Ajouter les valeurs sur les barres
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height()}%', ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height()}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('data/plot_terrain_neutre.png', dpi=150, bbox_inches='tight')
plt.show()

---
## ⚽ 6. Analyse des buts

On analyse les buts : combien par match, à quelle minute, combien de penaltys et de CSC.  
Ces infos vont servir à construire nos features de style de jeu.

In [ ]:
# ── Statistiques générales sur les buts ───────────────
r90['total_goals'] = r90.home_score + r90.away_score  # total buts par match

print('=== STATISTIQUES BUTS (post-1990) ===')
print(f'Moyenne buts/match    : {r90.total_goals.mean():.2f}')
print(f'Buts domicile moy.    : {r90.home_score.mean():.2f}')
print(f'Buts extérieur moy.   : {r90.away_score.mean():.2f}')
print(f'Max buts dans un match: {r90.total_goals.max():.0f}')
print()

# Distribution des scores les plus fréquents
print('Top 10 scores les plus fréquents :')
score_dist = r90.groupby(['home_score', 'away_score']).size().sort_values(ascending=False).head(10)
print(score_dist)

In [ ]:
# ── Analyse des minutes de buts ───────────────────────
# On filtre sur minute.notna() car 256 buts n'ont pas de minute renseignée

g90 = goals[
    (goals.date.dt.year >= 1990) &
    (goals.minute.notna())             # on garde seulement les buts avec une minute
].copy()

print(f'Buts avec minute renseignée : {len(g90):,}')
print()

# Découper en intervalles de 15 minutes
bins   = [0, 15, 30, 45, 60, 75, 90, 150]
labels = ['0-15', '15-30', '30-45', '45-60', '60-75', '75-90', '90+']
g90['period'] = pd.cut(g90.minute, bins=bins, labels=labels)

print('Buts par période de 15 minutes :')
period_counts = g90.period.value_counts().sort_index()
print(period_counts)
print()
print('💡 Insight : 75-90 min concentre 21% des buts — les équipes marquent plus en fin de match')

In [ ]:
# ── Graphique : buts par période ──────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

colors_bars = ['#BDD7EE', '#9DC3E6', '#2E75B6', '#1F4E79', '#1F4E79', '#0D2137', '#95A5A6']
bars = ax.bar(labels, period_counts.values, color=colors_bars, edgecolor='white', linewidth=0.5)

ax.set_title('Distribution des buts par période de 15 minutes (post-1990)', fontsize=13, fontweight='bold')
ax.set_xlabel('Période du match', fontsize=11)
ax.set_ylabel('Nombre de buts', fontsize=11)

# Ajouter les valeurs
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{bar.get_height():,}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('data/plot_buts_periodes.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Penaltys et CSC ───────────────────────────────────
# Ces stats vont servir à créer des features de style de jeu

print('=== PENALTYS ET CSC (post-1990) ===')
print()

total_buts = len(g90)
nb_pen = g90.penalty.sum()    # nombre de buts sur penalty
nb_csc = g90.own_goal.sum()   # nombre de buts contre son camp

print(f'Total buts analysés : {total_buts:,}')
print(f'Penaltys            : {nb_pen:,} ({nb_pen/total_buts:.1%} des buts)')
print(f'CSC (contre son camp): {nb_csc:,} ({nb_csc/total_buts:.1%} des buts)')
print()
print('💡 Insight : 6.8% des buts viennent de penaltys')
print('   → Le taux de penaltys obtenus/concédés sera une feature utile')

---
## 🏆 7. Analyse par tournoi

On compare les résultats selon le type de compétition.  
La CDM se comporte différemment des amicaux ou des qualifications.

In [ ]:
# ── Résultats par grand tournoi ───────────────────────
tournois = {
    'FIFA World Cup'          : 'CDM',
    'Friendly'                : 'Amical',
    'UEFA Euro'               : 'Euro',
    'Copa América'            : 'Copa América',
    'African Cup of Nations'  : 'CAN',
    'UEFA Nations League'     : 'Nations League'
}

print('=== RÉSULTATS PAR TOURNOI (post-1990) ===')
print(f'{"Tournoi":<20} {"Matchs":>8} {"Home Win":>10} {"Draw":>8} {"Away Win":>10} {"Buts/Match":>12}')
print('-' * 70)

for tournoi, label in tournois.items():
    sub = r90[r90.tournament == tournoi]
    if len(sub) == 0:
        continue
    res  = sub.result.value_counts(normalize=True)
    buts = sub.total_goals.mean()
    print(
        f'{label:<20}'
        f'{len(sub):>8,}'
        f'{res.get("home_win",0):>10.1%}'
        f'{res.get("draw",0):>8.1%}'
        f'{res.get("away_win",0):>10.1%}'
        f'{buts:>12.2f}'
    )

print()
print('💡 Insight : La CDM produit plus de victoires extérieures (31.7%) que les amicaux (25.4%)')
print('   → Les matchs à élimination directe sont plus ouverts et imprévisibles')

In [ ]:
# ── Top tournois par nombre de matchs ─────────────────
print('Top 15 tournois dans le dataset :')
print(results.tournament.value_counts().head(15))

---
## 🎯 8. Focus CDM 2026

On analyse spécifiquement les 72 matchs CDM 2026 qui sont dans notre dataset.  
Ce sont exactement les matchs à prédire — leurs scores sont à NaN.

In [ ]:
# ── Isoler les matchs CDM 2026 ────────────────────────
# Ce sont les lignes avec home_score = NaN dans results.csv

cdm2026 = results[
    (results.tournament == 'FIFA World Cup') &
    (results.date.dt.year == 2026)
].copy()

print(f'Matchs CDM 2026 : {len(cdm2026)}')
print(f'Tous sur terrain neutre : {cdm2026.neutral.all()}')  # True = tous neutral
print()
print('Dates des matchs :')
print(f'  Premier match : {cdm2026.date.min().date()}')
print(f'  Dernier match : {cdm2026.date.max().date()}')
print()
print('Équipes présentes :')
teams = sorted(set(cdm2026.home_team.tolist() + cdm2026.away_team.tolist()))
print(f'{len(teams)} équipes : {teams}')

In [ ]:
# ── Afficher les 72 matchs à prédire ──────────────────
print('Les 72 matchs CDM 2026 à prédire :')
cdm2026[['date', 'home_team', 'away_team', 'neutral']].to_string(index=False)

---
## 🔍 9. Détection des anomalies

On cherche les problèmes dans le dataset : doublons, scores aberrants, incohérences.

In [ ]:
# ── Doublons ──────────────────────────────────────────
# Un doublon = même date, mêmes équipes, mais scores différents

doublons = results[results.duplicated(subset=['date', 'home_team', 'away_team'], keep=False)]
print(f'Doublons trouvés : {len(doublons)}')
if len(doublons) > 0:
    print(doublons.to_string())

In [ ]:
# ── Scores aberrants ──────────────────────────────────
# On cherche les matchs avec des scores très élevés (> 20 buts)
# Ces matchs sont réels mais peuvent biaiser nos features

r_clean = results.dropna(subset=['home_score'])
outliers = r_clean[(r_clean.home_score > 20) | (r_clean.away_score > 20)]

print(f'Matchs avec score > 20 buts : {len(outliers)}')
print()
print(outliers[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']].to_string())
print()
print('💡 Insight : Ces matchs sont réels (ex: Australie 31-0 Samoa américaines)')
print('   → On les garde mais on cappera les features de buts au percentile 99')

In [ ]:
# ── Vérification former_names ─────────────────────────
# On vérifie si des anciens noms subsistent encore dans results.csv

all_teams = set(results.home_team.unique()) | set(results.away_team.unique())
anciens_noms_presents = []

for _, row in former.iterrows():
    if row['former'] in all_teams:
        anciens_noms_presents.append(row['former'])

if len(anciens_noms_presents) == 0:
    print('✅ Aucun ancien nom trouvé — le dataset est déjà normalisé')
else:
    print(f'⚠️ Anciens noms encore présents : {anciens_noms_presents}')

---
## 📈 10. Évolution temporelle

On regarde comment le football a évolué au fil des décennies.  
Ça justifie notre choix de ne garder que les matchs post-1990.

In [ ]:
# ── Évolution par décennie ────────────────────────────
r_all = results.dropna(subset=['home_score']).copy()
r_all['decade'] = (r_all.date.dt.year // 10) * 10   # arrondir à la décennie
r_all['result'] = np.where(
    r_all.home_score > r_all.away_score, 'home_win',
    np.where(r_all.home_score < r_all.away_score, 'away_win', 'draw')
)
r_all['total_goals'] = r_all.home_score + r_all.away_score

# Agréger par décennie
decade_stats = r_all.groupby('decade').agg(
    nb_matchs    = ('result', 'count'),
    home_win_pct = ('result', lambda x: (x == 'home_win').mean()),
    draw_pct     = ('result', lambda x: (x == 'draw').mean()),
    away_win_pct = ('result', lambda x: (x == 'away_win').mean()),
    avg_goals    = ('total_goals', 'mean')
).round(3)

print('Statistiques par décennie :')
print(decade_stats.to_string())

In [ ]:
# ── Graphique : évolution des buts par décennie ───────
fig, ax = plt.subplots(figsize=(12, 5))

decades = decade_stats.index[decade_stats.index >= 1930]  # depuis 1930
avg_goals_filtered = decade_stats.loc[decades, 'avg_goals']

ax.bar(decades.astype(str), avg_goals_filtered.values, color='#2E75B6', alpha=0.85, edgecolor='white')
ax.axvline(x='1990', color='red', linestyle='--', linewidth=2, label='Filtre post-1990')

ax.set_title('Évolution de la moyenne de buts par match (par décennie)', fontsize=13, fontweight='bold')
ax.set_xlabel('Décennie', fontsize=11)
ax.set_ylabel('Buts par match', fontsize=11)
ax.legend(fontsize=11)

for i, (dec, val) in enumerate(zip(decades, avg_goals_filtered.values)):
    ax.text(i, val + 0.05, f'{val:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('data/plot_evolution_buts.png', dpi=150, bbox_inches='tight')
plt.show()
print('💡 Insight : Le football moderne (post-1990) est différent — moins de buts, plus tactique')
print('   → On entraîne le modèle uniquement sur les données post-1990')

---
## 📋 11. Résumé des insights EDA

Ce qu'on a appris et ce que ça implique pour la suite.

In [ ]:
# ── Résumé complet ────────────────────────────────────
print('=' * 60)
print('RÉSUMÉ EDA — INSIGHTS CLÉS')
print('=' * 60)
print()
print('📊 DATASET')
print(f'   {len(results):,} matchs au total (1872–2026)')
print(f'   {len(r90):,} matchs post-1990 (notre set d\'entraînement)')
print(f'   72 matchs CDM 2026 à prédire (scores NaN)')
print()
print('🎯 TARGET (variable à prédire)')
print('   home_win : 48.3%  ← classe majoritaire')
print('   away_win : 28.0%')
print('   draw     : 23.7%  ← classe minoritaire')
print('   → Déséquilibre à gérer avec class_weight="balanced"')
print()
print('🏟️ TERRAIN NEUTRE')
print('   Terrain réel → home_win : 50.7%')
print('   Terrain neutre → home_win : 42.6%')
print('   → Feature critique : colonne neutral')
print('   → CDM 2026 = 100% terrain neutre')
print()
print('⚽ BUTS')
print('   Moyenne : 2.76 buts/match')
print('   21% des buts tombent entre 75-90 min')
print('   6.8% des buts sont des penaltys')
print()
print('🔧 NETTOYAGE NÉCESSAIRE')
print('   1. Isoler les 72 matchs CDM 2026')
print('   2. Filtrer post-1990')
print('   3. Supprimer le doublon Tahiti-Nouvelle Calédonie')
print('   4. Former names : déjà normalisé ✅')
print('   5. Cap features buts au percentile 99')
print()
print('✅ PROCHAINE ÉTAPE : 01_cleaning.py')